In [2]:
import rasterio
import numpy as np
import pandas as pd
import geopandas as gpd
import os
import rasterio

from pathlib import Path
from typing import List, Tuple, Optional, Union
from rasterio.enums import MergeAlg
from rasterio import features

import sys
if sys.version_info < (3, 9):
    from importlib_resources import files
else:
    from importlib.resources import files

from beak.utilities.conversions import create_geodataframe_from_points, create_geodataframe_from_polygons, _rasterize_vector_process, create_binary_raster
from beak.utilities.io import save_raster, check_path, load_raster, load_dataset


# Load data

**User definitions**

In [3]:
# Set base paths and files
BASE_PATH = files("beak.data")
PATH_BASE_RASTER = BASE_PATH / "BASE_RASTERS" / "EPSG_102008_RES_500_MVT_CE.tif"

# Points file and query to select relevant mineral occurences
PATH_LABELS = BASE_PATH / "RAW" / "mineral_deposits" / "Mississippy_Valley_Type" / "TA2_McCafferty_CEUS_splits_from_hack_9m" / "train.csv"

# Set the output file
PATH_EXPORT = BASE_PATH / "TRAINING_LABELS" / "MVT_REG_CEUS" / str(PATH_BASE_RASTER.stem + "_CEUS_MCCAFFERTY_TRAIN_80" + ".tif")

# Overwrite the path for testing purposes
CURRENT_DIR = Path(os.getcwd())
OUT_FILE = CURRENT_DIR / PATH_EXPORT.name
OUT_FILE = PATH_EXPORT

print(f"Output file: {OUT_FILE}")

Output file: s:\projekte\20230082_darpa_criticalmaas_ta3\bearbeitung\github\beak-ta3\src\beak\data\TRAINING_LABELS\MVT_REG_CEUS\EPSG_102008_RES_500_MVT_CE_CEUS_MCCAFFERTY_TRAIN_80.tif


In [4]:
mineral_sites = load_dataset(PATH_LABELS)
mineral_sites.head()

,x,y,label,lon,lat,source
0,516,1812,1,100750,-355750,0
1,511,1812,1,98250,-355750,0
2,521,1805,1,103250,-352250,0
3,525,1801,1,105250,-350250,0
4,518,1814,1,101750,-356750,0


# Create Labels

In [5]:
base_raster = rasterio.open(PATH_BASE_RASTER)


In [6]:
data = mineral_sites.copy()
gdf = create_geodataframe_from_points(data, long_col="lon", lat_col="lat", crs=base_raster.meta["crs"])
gdf.head()

,x,y,label,lon,lat,source,geometry
0,516,1812,1,100750,-355750,0,POINT (100750.000 -355750.000)
1,511,1812,1,98250,-355750,0,POINT (98250.000 -355750.000)
2,521,1805,1,103250,-352250,0,POINT (103250.000 -352250.000)
3,525,1801,1,105250,-350250,0,POINT (105250.000 -350250.000)
4,518,1814,1,101750,-356750,0,POINT (101750.000 -356750.000)


In [7]:
labels_array = create_binary_raster(gdf, base_raster, all_touched=False, same_shape=True, fill_negatives=True, out_file=PATH_EXPORT)